# 19 · LSTM 与序列任务

> **本节属于 Part 7 · 序列模型 RNN/LSTM。Part 7 收尾。**

上一节我们看到 vanilla RNN 因梯度消失而学不动长依赖。**LSTM** 通过一条"记忆细胞 (cell state)"的传送带和三个**门**（输入/遗忘/输出），为梯度开辟了一条畅通的"高速公路"，从而能记住很久以前的信息。本节实现 LSTM，并让它在上一节 RNN 失败的长序列任务上**完胜**。

## 学习目标

- 理解 LSTM 的**门控机制**与**记忆细胞**，以及它为何能缓解梯度消失
- 实现 `LSTMCell` / `LSTM`，并了解"遗忘门偏置初始化为 1"的重要技巧
- 在长序列记忆任务上对比 LSTM 与 RNN
- 用 LSTM 做一个**字符级生成**小demo

## LSTM 的门控机制

LSTM 维护两条状态：隐藏态 $h_t$ 和记忆细胞 $c_t$。三个门（值域 0~1）控制信息流：

$$i=\sigma(\cdot)\ \text{输入门},\quad f=\sigma(\cdot)\ \text{遗忘门},\quad o=\sigma(\cdot)\ \text{输出门},\quad g=\tanh(\cdot)\ \text{候选记忆}$$
$$c_t = f\odot c_{t-1} + i\odot g,\qquad h_t = o\odot\tanh(c_t)$$

关键在 $c_t = f\odot c_{t-1} + \dots$ 这个**加法**更新：当遗忘门 $f\approx 1$ 时，$c_{t-1}$ 几乎原样传到 $c_t$，梯度也就能几乎无损地往回流——这就是缓解梯度消失的"高速公路"。

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad
from minitorch.optim import Adam
from minitorch.functional import cross_entropy

print(inspect.getsource(nn.LSTMCell.forward))

> **小技巧**：我们把**遗忘门的偏置初始化为 1**（而非 0）。这样训练初期遗忘门接近"全开"，LSTM 默认倾向于"记住"，更容易学到长依赖。这是实践中非常重要的一个细节。

## 长依赖任务：LSTM vs RNN

还是上一节的"记忆第一个 token"任务，但用 **T=50** 的长序列——上一节 RNN 在这退化成了随机猜。

In [ ]:
def make_memory(n, T, seed):
    rng = np.random.RandomState(seed)
    sig = rng.choice([-1.0, 1.0], size=(n, 1))
    noise = rng.randn(n, T - 1) * 0.5
    X = np.concatenate([sig, noise], axis=1)[:, :, None]
    y = (sig[:, 0] > 0).astype(int)
    return X, y

def train(kind, T=50, epochs=80):
    minitorch.set_seed(0)
    rec = nn.RNN(1, 32) if kind == "RNN" else nn.LSTM(1, 32)
    head = nn.Linear(32, 2)
    opt = Adam(rec.parameters() + head.parameters(), lr=5e-3)
    loss_fn = nn.CrossEntropyLoss()
    Xtr, ytr = make_memory(400, T, 0)
    for ep in range(epochs):
        opt.zero_grad()
        out = rec(Tensor(Xtr)); h_N = out[1] if kind == "RNN" else out[1][0]
        loss_fn(head(h_N), ytr).backward()
        opt.step()
    Xte, yte = make_memory(400, T, 1)
    out = rec(Tensor(Xte)); h_N = out[1] if kind == "RNN" else out[1][0]
    with no_grad():
        return (head(h_N).data.argmax(1) == yte).mean()

acc_rnn = train("RNN"); acc_lstm = train("LSTM")
print(f"T=50 长序列记忆任务：")
print(f"  vanilla RNN 准确率 = {acc_rnn*100:5.1f}%   （梯度消失，失败）")
print(f"  LSTM        准确率 = {acc_lstm*100:5.1f}%   （门控记忆，成功）")

**LSTM 完胜**：同样的长序列任务，RNN 退化到随机水平，而 LSTM 凭借门控记忆稳稳解决。

## 字符级生成小demo

让 LSTM 学习预测一句话里的"下一个字符"，然后从首字符开始**自回归生成**，看它能否复现这句话。这其实就是一个最小的"语言模型"。

In [ ]:
text = "hello minitorch and deep learning! "
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V = len(chars)
seq = [stoi[c] for c in text]

def one_hot(idxs):
    m = np.zeros((len(idxs), V)); m[np.arange(len(idxs)), idxs] = 1.0
    return m

minitorch.set_seed(0)
lstm = nn.LSTM(V, 64); head = nn.Linear(64, V)
opt = Adam(lstm.parameters() + head.parameters(), lr=8e-3)

X = one_hot(seq[:-1])[None, :, :]      # (1, T, V)
targets = np.array(seq[1:])
for ep in range(200):
    opt.zero_grad()
    outputs, _ = lstm(Tensor(X))
    loss = None
    for t, h_t in enumerate(outputs):
        l = cross_entropy(head(h_t), targets[t:t+1])
        loss = l if loss is None else loss + l
    (loss * (1.0 / len(outputs))).backward()
    opt.step()
print(f"训练完成，最终平均损失 {float(loss.data)/len(outputs):.4f}")

In [ ]:
# 自回归生成：从首字符出发，每次喂入上一步的预测
h = Tensor(np.zeros((1, 64))); c = Tensor(np.zeros((1, 64)))
idx = stoi[text[0]]; out_chars = [text[0]]
with no_grad():
    for _ in range(len(text) - 1):
        h, c = lstm.cell(Tensor(one_hot([idx])), (h, c))
        idx = int(head(h).data.argmax())
        out_chars.append(itos[idx])
print("目标:", repr(text))
print("生成:", repr("".join(out_chars)))

## PyTorch 对照

`nn.LSTM` 概念一致（返回整段输出与 (h_n, c_n)）。LSTM 至今仍是许多序列任务的强基线。

In [ ]:
import torch
tlstm = torch.nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
x = torch.randn(2, 5, 8)
out, (h_n, c_n) = tlstm(x)
print("PyTorch LSTM 输出:", tuple(out.shape), " h_n:", tuple(h_n.shape), " c_n:", tuple(c_n.shape))

## 📦 沉淀进 minitorch

`LSTMCell / LSTM` 在 `minitorch/nn/rnn.py`，由 `tests/test_rnn.py` 守护。

## 小练习

1. **遗忘门偏置**：把遗忘门偏置改回 0（默认值），重跑长依赖任务，LSTM 还那么强吗？体会这个技巧的作用。
2. **更长序列**：把 T 加到 100，LSTM 还能解决吗？需要更多隐藏单元或训练轮数吗？
3. **生成更长文本**：用一小段真实文字（如一句诗）训练字符级 LSTM，观察生成质量。

## 小结 & 下一站

✅ 我们实现了 LSTM，理解了门控如何打通梯度"高速公路"，并在 RNN 失败的长序列任务上取得成功，还做了一个字符级生成 demo。**Part 7 完成！**

**下一站 → Part 8 `20_attention_from_scratch`**：进入当前最重要的架构——**注意力机制与 Transformer**。RNN 必须一步步顺序处理序列，而注意力让模型**一步到位**地关注序列中任意位置，这正是大模型时代的基石。